# PRODUCTION - local or GCP

In [82]:
# ==== IMPORTS ====
from __future__ import annotations

import json
import logging
import re
import pandas as pd
from datetime import datetime

from pathlib import Path
from typing import Optional

import dateparser
import pdfplumber
from requests.adapters import HTTPAdapter

try:
    from google.cloud import storage
except Exception:
    storage = None

In [83]:
# ==== CONFIGURATION ====
MAX_PAGES = 4
MAX_TRIES = 3
MAX_TAIL_PAGES = 4
OUTPUT_DIR = Path.cwd()
OUTPUT_CSV = OUTPUT_DIR / "handin_month_summary.csv"
OUTPUT_JSON = OUTPUT_DIR / "handin_month_summary.json"
GCP_DEFAULT_BUCKET = "thesis_archive_bucket"
GCP_DEFAULT_PREFIX = "dtu_findit/master_thesis/"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger("seasonality_extractor")

# Danish -> English normalization so output is always "Month YYYY" in English
MONTH_TRANSLATIONS = {
    "januar": "january",
    "februar": "february",
    "marts": "march",
    "april": "april",
    "maj": "may",
    "juni": "june",
    "juli": "july",
    "august": "august",
    "september": "september",
    "oktober": "october",
    "november": "november",
    "december": "december",
}

# Abbreviation normalization (English + Danish), optional trailing dot handled in normalization.
MONTH_ABBR_TRANSLATIONS = {
    "jan": "january",
    "feb": "february",
    "mar": "march",
    "apr": "april",
    "may": "may",
    "maj": "may",
    "jun": "june",
    "jul": "july",
    "aug": "august",
    "sep": "september",
    "sept": "september",
    "oct": "october",
    "okt": "october",
    "nov": "november",
    "dec": "december",
}

# Constrain textual month-year matches to actual month words (full names + common abbreviations).
MONTH_NAME_REGEX = (
    r"(?:jan(?:uary)?|feb(?:ruary)?|mar(?:ch|ts)?|apr(?:il)?|may|maj|jun(?:e|i)?|"
    r"jul(?:y|i)?|aug(?:ust)?|sep(?:t(?:ember)?)?|oct(?:ober)?|okt(?:ober)?|"
    r"nov(?:ember)?|dec(?:ember)?)"
)

# Ordered by confidence: ranges first (must take latter date), then explicit month-year patterns.
PATTERNS = [
    # 29/10/2018-29/04/2019 (extract latter date)
    re.compile(
        r"\b"
        r"(\d{1,2}[./-]\d{1,2}[./-]\d{2,4})"
        r"\s*(?:-|–|—|to)\s*"
        r"(\d{1,2}[./-]\d{1,2}[./-]\d{2,4})"
        r"\b",
        re.IGNORECASE,
    ),
    # January 2019 to June 2019 / Jan 2019 - Jun 2019 (extract latter month-year)
    re.compile(
        rf"\b({MONTH_NAME_REGEX}\s*(?:of\s+)?(?:'\d{{2}}|\d{{2,4}}))"
        rf"\s*(?:-|–|—|to|until|til)\s*"
        rf"({MONTH_NAME_REGEX}\s*(?:of\s+)?(?:'\d{{2}}|\d{{2,4}}))\b",
        re.IGNORECASE,
    ),
    # 2019 June to 2020 July / 19 Jun - 20 Jul (extract latter year-month)
    re.compile(
        rf"\b((?:'\d{{2}}|\d{{2,4}})\s*{MONTH_NAME_REGEX})"
        rf"\s*(?:-|–|—|to|until|til)\s*"
        rf"((?:'\d{{2}}|\d{{2,4}})\s*{MONTH_NAME_REGEX})\b",
        re.IGNORECASE,
    ),
    # 27January2025to27July2025 / 27thofJune,2020to27thofJuly,2020 (extract latter date)
    re.compile(
        r"\b"
        r"(\d{1,2}\s*(?:st|nd|rd|th)?\s*(?:of(?:\s+|[-/.])?)?[A-Za-zÀ-ÖØ-öø-ÿ]+\s*,?\s*\d{2,4})"
        r"\s*(?:-|–|—|to)\s*"
        r"(\d{1,2}\s*(?:st|nd|rd|th)?\s*(?:of(?:\s+|[-/.])?)?[A-Za-zÀ-ÖØ-öø-ÿ]+\s*,?\s*\d{2,4})"
        r"\b",
        re.IGNORECASE,
    ),
    # Compact numeric dates: YYYYMMDD or DDMMYYYY
    re.compile(r"\b(\d{8})\b"),
    # YYYY-MM-DD or YYYY-DD-MM with flexible separators
    re.compile(r"\b(\d{4}[-/.]\d{1,2}[-/.]\d{1,2})\b"),
    # Location, DD-Month-YYYY (e.g., Lund, 10-July-2019)
    re.compile(
        r"\b(?:[^,\n]{1,80})\s*,\s*"
        r"(\d{1,2}[-/.]\s*[A-Za-zÀ-ÖØ-öø-ÿ]+\s*[-/.]?\s*\d{2,4})\b",
        re.IGNORECASE,
    ),
    # DD-Month-YYYY, Location (e.g., 10-July-2019, Lund)
    re.compile(
        r"\b(\d{1,2}[-/.]\s*[A-Za-zÀ-ÖØ-öø-ÿ]+\s*[-/.]?\s*\d{2,4})"
        r"\s*,\s*[^,\n]{1,80}\b",
        re.IGNORECASE,
    ),
    # DD-Month-YYYY or DD/Month/YYYY (e.g., 10-June-2019, 10/June/2019)
    re.compile(
        r"\b"
        r"(\d{1,2}[-/.]\s*[A-Za-zÀ-ÖØ-öø-ÿ]+\s*[-/.]?\s*\d{2,4})"
        r"\b",
        re.IGNORECASE,
    ),
    # Month-DD-YYYY or Month/DD/YYYY (e.g., June-10-2019, June/10/2019)
    re.compile(
        r"\b"
        r"([A-Za-zÀ-ÖØ-öø-ÿ]+\s*[-/.]\s*\d{1,2}\s*[-/.]\s*\d{2,4})"
        r"\b",
        re.IGNORECASE,
    ),
    # June 2, 2019 / JUNE 2, 2019
    re.compile(
        r"\b"
        r"((?:[A-Za-zÀ-ÖØ-öø-ÿ]+)\s+\d{1,2},?\s+\d{2,4})"
        r"\b",
        re.IGNORECASE,
    ),
    # June 28th, 2019 (ordinal suffix)
    re.compile(
        r"\b"
        r"([A-Za-zÀ-ÖØ-öø-ÿ]+\s+\d{1,2}(?:st|nd|rd|th),?\s+\d{2,4})"
        r"\b",
        re.IGNORECASE,
    ),
    # 27th of June, 2020 / 27thofJune,2020 / 27th-of-June-2020 / 27 June 2020
    re.compile(
        r"\b"
        r"(\d{1,2}\s*(?:st|nd|rd|th)?\s*(?:of(?:\s+|[-/.])?)?[A-Za-zÀ-ÖØ-öø-ÿ]+\s*,?\s*\d{2,4})"
        r"\b",
        re.IGNORECASE,
    ),
    # Month of Year (e.g., June of 2019, sep of '19)
    re.compile(
        rf"\b({MONTH_NAME_REGEX}\s+of\s+(?:'\d{{2}}|\d{{2,4}}))\b",
        re.IGNORECASE,
    ),
    # 2 June 2019
    re.compile(
        r"\b"
        r"(\d{1,2}\s+[A-Za-zÀ-ÖØ-öø-ÿ]+\s+\d{2,4})"
        r"\b",
        re.IGNORECASE,
    ),
    # Month-YYYY / Month/YYYY / Month.YYYY / Month, YYYY / Month YYYY / Month 'YY
    re.compile(
        rf"\b({MONTH_NAME_REGEX}\s*(?:,|[-/.])?\s*(?:'\d{{2}}|\d{{2,4}}))\b",
        re.IGNORECASE,
    ),
    # YYYY Month / YYYY-Month / YYYY/Month / YYYY.Month / 'YY Month
    re.compile(
        rf"\b((?:'\d{{2}}|\d{{2,4}})\s*(?:[-/.])?\s*{MONTH_NAME_REGEX})\b",
        re.IGNORECASE,
    ),
    # MM YYYY / MM-YYYY / MM/YYYY / MM.YYYY (e.g., 05 2020, 05-2020)
    re.compile(r"\b((?:0?[1-9]|1[0-2])(?:\s+|[-/.])\d{4})\b"),
    # YYYY-MM / YYYY/MM / YYYY.MM / YYYY MM (e.g., 2020-05, 2020 05)
    re.compile(r"\b(\d{4}(?:\s+|[-/.])(?:0?[1-9]|1[0-2]))\b"),
    # Numeric single date fallback
    re.compile(r"\b(\d{1,2}[./-]\d{1,2}[./-]\d{2,4})\b"),
]

In [84]:
# ==== FUNCTIONS ====
CID_DENSITY_THRESHOLD = globals().get("CID_DENSITY_THRESHOLD", 0.05)


def normalize_text(text: str) -> str:
    """Lower-risk normalization to improve pattern matching and month parsing."""
    cleaned = text.replace("\u00a0", " ")
    cleaned = re.sub(r"[‐‑‒–—−]", "-", cleaned)
    cleaned = re.sub(r"\s+", " ", cleaned)
    cleaned = re.sub(r"\s*,\s*", ", ", cleaned)
    cleaned = re.sub(
        r"(?<=\d)(?=(?!st\b|nd\b|rd\b|th\b)[A-Za-zÀ-ÖØ-öø-ÿ])",
        " ",
        cleaned,
        flags=re.IGNORECASE,
    )
    cleaned = re.sub(
        r"(?i)(\d{1,2}\s*(?:st|nd|rd|th))\s*of(?=[A-Za-zÀ-ÖØ-öø-ÿ])",
        r"\1 of ",
        cleaned,
    )
    cleaned = re.sub(r"(?i)\bof(?=[A-Za-zÀ-ÖØ-öø-ÿ])", "of ", cleaned)
    cleaned = re.sub(r"(?<=\d)to(?=\d)", " to ", cleaned, flags=re.IGNORECASE)

    # Repair OCR-glued month/day dates like "April23,2019" -> "April 23, 2019".
    cleaned = re.sub(r"(?<=[A-Za-zÀ-ÖØ-öø-ÿ])(?=\d)", " ", cleaned)
    cleaned = re.sub(r"(?<=\d),(?=\d{2,4}\b)", ", ", cleaned)

    # Normalize month abbreviations and dotted forms: Sep. -> september, okt. -> october
    for abbr, full in MONTH_ABBR_TRANSLATIONS.items():
        cleaned = re.sub(rf"\b{abbr}\.?\b", full, cleaned, flags=re.IGNORECASE)

    # Remove leftover month-period punctuation in constructs like "October. 12".
    cleaned = re.sub(r"\b([A-Za-zÀ-ÖØ-öø-ÿ]+)\.(?=\s*\d)", r"\1", cleaned)

    # Replace Danish month names with English names for consistent parsing/output.
    for dk, en in MONTH_TRANSLATIONS.items():
        cleaned = re.sub(rf"\b{dk}\b", en, cleaned, flags=re.IGNORECASE)

    return cleaned.strip()


def expand_two_digit_year(year_text: str) -> int:
    """Convert 2-digit year to 4-digit year using strptime semantics (69-99 -> 1900s, 00-68 -> 2000s)."""
    year_text = year_text.strip().lstrip("'")
    if len(year_text) == 2:
        return datetime.strptime(year_text, "%y").year
    return int(year_text)


def parse_to_month_year(date_text: str) -> str | None:
    """Parse a candidate date string to standardized 'Month YYYY'."""
    date_text = date_text.strip()

    # Compact numeric dates: YYYYMMDD or DDMMYYYY.
    yyyymmdd = re.fullmatch(r"(\d{4})(\d{2})(\d{2})", date_text)
    if yyyymmdd:
        year = int(yyyymmdd.group(1))
        month = int(yyyymmdd.group(2))
        day = int(yyyymmdd.group(3))
        try:
            return datetime(year, month, day).strftime("%B %Y")
        except ValueError:
            pass

    ddmmyyyy = re.fullmatch(r"(\d{2})(\d{2})(\d{4})", date_text)
    if ddmmyyyy:
        day = int(ddmmyyyy.group(1))
        month = int(ddmmyyyy.group(2))
        year = int(ddmmyyyy.group(3))
        try:
            return datetime(year, month, day).strftime("%B %Y")
        except ValueError:
            pass

    # Explicit numeric month-year formats.
    mm_yyyy = re.fullmatch(r"(0?[1-9]|1[0-2])(?:\s+|[-/.])(\d{4})", date_text)
    if mm_yyyy:
        month = int(mm_yyyy.group(1))
        year = int(mm_yyyy.group(2))
        return datetime(year, month, 1).strftime("%B %Y")

    yyyy_mm = re.fullmatch(r"(\d{4})(?:\s+|[-/.])(0?[1-9]|1[0-2])", date_text)
    if yyyy_mm:
        year = int(yyyy_mm.group(1))
        month = int(yyyy_mm.group(2))
        return datetime(year, month, 1).strftime("%B %Y")

    # Explicit textual month-year formats in both orders, including "month of year" and 2-digit years.
    month_yyyy = re.fullmatch(
        rf"({MONTH_NAME_REGEX})\s*(?:,|[-/.])?\s*(?:of\s+)?('?\d{{2,4}})",
        date_text,
        flags=re.IGNORECASE,
    )
    if month_yyyy:
        month_name = month_yyyy.group(1)
        year = expand_two_digit_year(month_yyyy.group(2))
        month_dt = dateparser.parse(
            month_name,
            languages=["en", "da"],
            settings={"PREFER_DAY_OF_MONTH": "first", "NORMALIZE": True},
        )
        if month_dt:
            return datetime(year, month_dt.month, 1).strftime("%B %Y")

    yyyy_month = re.fullmatch(
        rf"('?\d{{2,4}})\s*(?:[-/.])?\s*({MONTH_NAME_REGEX})",
        date_text,
        flags=re.IGNORECASE,
    )
    if yyyy_month:
        year = expand_two_digit_year(yyyy_month.group(1))
        month_name = yyyy_month.group(2)
        month_dt = dateparser.parse(
            month_name,
            languages=["en", "da"],
            settings={"PREFER_DAY_OF_MONTH": "first", "NORMALIZE": True},
        )
        if month_dt:
            return datetime(year, month_dt.month, 1).strftime("%B %Y")

    # Detect ISO-like year-first format and parse with MDY to keep year-first interpretation.
    if re.match(r"\d{4}[-/.]\d{1,2}[-/.]\d{1,2}", date_text):
        settings = {
            "DATE_ORDER": "MDY",
            "PREFER_DAY_OF_MONTH": "first",
            "PREFER_DATES_FROM": "past",
            "NORMALIZE": True,
        }
    else:
        settings = {
            "DATE_ORDER": "DMY",
            "PREFER_DAY_OF_MONTH": "first",
            "PREFER_DATES_FROM": "past",
            "NORMALIZE": True,
        }

    dt = dateparser.parse(
        date_text,
        languages=["en", "da"],
        settings=settings,
    )
    if not dt:
        return None
    return dt.strftime("%B %Y")


def extract_handin_month(text: str) -> str | None:
    normalized = normalize_text(text)

    for pattern in PATTERNS:
        for match in pattern.finditer(normalized):
            groups = [g for g in match.groups() if g]

            # Date range: always use latter date.
            if len(groups) == 2:
                candidate = groups[1]
                parsed = parse_to_month_year(candidate)
                if parsed:
                    return parsed
            else:
                candidate = groups[0] if groups else match.group(0)
                parsed = parse_to_month_year(candidate)
                if parsed:
                    return parsed

    return None


def calculate_cid_density(text: str) -> float:
    """Calculate approximate CID density from '(cid:NNN)' markers in text."""
    if not text:
        return 0.0
    cid_matches = re.findall(r"\(cid:\d+\)", text)
    if not cid_matches:
        return 0.0
    cid_chars = sum(len(m) for m in cid_matches)
    return cid_chars / max(len(text), 1)


def is_pdf_corrupt(pdf_path: Path) -> bool:
    """Mark PDF as corrupt if CID density on sampled pages exceeds threshold."""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            total_pages = len(pdf.pages)
            if total_pages == 0:
                return False

            sample_pages = min(3, total_pages)
            sample_text = "\n".join(
                (pdf.pages[i].extract_text() or "") for i in range(sample_pages)
            )
            return calculate_cid_density(sample_text) > CID_DENSITY_THRESHOLD
    except Exception as e:
        logger.error("Error checking CID corruption for %s: %s", pdf_path.name, e)
        return False


def extract_handin_month_from_pdf(
    pdf_path: Path,
    chunk_size: int = MAX_PAGES,
    max_tries: int = MAX_TRIES,
    tail_pages: int = MAX_TAIL_PAGES,
) -> str | None:
    """
    Scan PDF in chunk_size-page windows and stop after max_tries windows.
    If no date is found, also scan the last `tail_pages` pages where colophon dates
    are often located.

    If PDF is CID-corrupt (>5%), skip extraction and return None.
    """
    if is_pdf_corrupt(pdf_path):
        return None

    with pdfplumber.open(pdf_path) as pdf:
        total_pages = len(pdf.pages)

        for attempt_idx, start in enumerate(range(0, total_pages, chunk_size), start=1):
            if attempt_idx > max_tries:
                break

            chunk = pdf.pages[start : start + chunk_size]
            text = "\n".join((page.extract_text() or "") for page in chunk)
            parsed = extract_handin_month(text)
            if parsed:
                return parsed

        # Fallback: scan tail pages to catch date blocks placed at the end of thesis PDFs.
        if tail_pages > 0 and total_pages > 0:
            start_tail = max(0, total_pages - tail_pages)
            tail_chunk = pdf.pages[start_tail:total_pages]
            tail_text = "\n".join((page.extract_text() or "") for page in tail_chunk)
            parsed = extract_handin_month(tail_text)
            if parsed:
                return parsed

    return None


def run_extraction(pdf_files: list[Path]) -> list[dict[str, str | bool | None]]:
    results: list[dict[str, str | bool | None]] = []

    for pdf_file in pdf_files:
        corrupt_cid = is_pdf_corrupt(pdf_file)
        handin_month = None
        if not corrupt_cid:
            handin_month = extract_handin_month_from_pdf(
                pdf_file,
                chunk_size=MAX_PAGES,
                max_tries=MAX_TRIES,
            )

        results.append(
            {
                "filename": pdf_file.name,
                "handin_month": handin_month,
                "corrupt_cid": corrupt_cid,
            }
        )

    return results


class GCPBatchProcessor:
    """Process thesis PDFs in batches from GCS for seasonality extraction."""

    def __init__(
        self,
        bucket_name: str,
        gcs_prefix: str = "dtu_findit/master_thesis",
        local_pdf_folder: str = "thesis_pdfs",
        batch_size: int = 10,
    ):
        if storage is None:
            raise ImportError(
                "google-cloud-storage is not installed. Install it to use gcp mode."
            )

        self.bucket_name = bucket_name
        self.gcs_prefix = gcs_prefix
        self.local_pdf_folder = Path(local_pdf_folder)
        self.batch_size = batch_size

        self.local_pdf_folder.mkdir(exist_ok=True)

        logger.info("Initialising GCP Storage client...")
        self.client = storage.Client()
        self.bucket = self.client.bucket(bucket_name)
        self._configure_http_pool(max_pool_size=max(10, self.batch_size * 2))
        logger.info("Connected to bucket: gs://%s", bucket_name)

    def _configure_http_pool(self, max_pool_size: int) -> None:
        """Configure the underlying HTTP adapter pool size for GCS requests."""
        pool_size = max(10, int(max_pool_size))
        try:
            adapter = HTTPAdapter(
                pool_connections=pool_size,
                pool_maxsize=pool_size,
                max_retries=0,
            )
            self.client._http.mount("https://", adapter)
            self.client._http.mount("http://", adapter)
            logger.debug("Configured HTTP connection pool size: %d", pool_size)
        except Exception as e:
            logger.debug("Could not tune HTTP pool size: %s", e)

    def list_thesis_pdfs(self) -> list[str]:
        logger.info(f"Listing PDFs from gs://{self.bucket_name}/{self.gcs_prefix}/")
        blobs = self.client.list_blobs(self.bucket_name, prefix=self.gcs_prefix)
        pdf_paths = [blob.name for blob in blobs if blob.name.lower().endswith(".pdf")]
        logger.info(f"Found {len(pdf_paths)} PDF files")
        return pdf_paths

    def download_batch(self, pdf_paths: list[str], start_idx: int) -> list[Path]:
        end_idx = min(start_idx + self.batch_size, len(pdf_paths))
        batch = pdf_paths[start_idx:end_idx]

        logger.info(
            f"Downloading batch {start_idx // self.batch_size + 1}: "
            f"PDFs {start_idx + 1}-{end_idx} of {len(pdf_paths)}"
        )

        local_paths: list[Path] = []
        for gcs_path in batch:
            filename = Path(gcs_path).name
            local_path = self.local_pdf_folder / filename

            try:
                blob = self.bucket.blob(gcs_path)
                blob.download_to_filename(str(local_path))
                local_paths.append(local_path)
                logger.info(f"  Downloaded: {filename}")
            except Exception as e:
                logger.error(f"  Failed to download {filename}: {e}")

        return local_paths

    def extract_from_batch(self, pdf_paths: list[Path]) -> list[dict[str, str | bool | None]]:
        results: list[dict[str, str | bool | None]] = []
        logger.info(f"Extracting hand-in month from {len(pdf_paths)} PDFs...")

        for pdf_path in pdf_paths:
            try:
                corrupt_cid = is_pdf_corrupt(pdf_path)
                handin_month = None
                if not corrupt_cid:
                    handin_month = extract_handin_month_from_pdf(
                        pdf_path,
                        chunk_size=MAX_PAGES,
                        max_tries=MAX_TRIES,
                    )

                results.append(
                    {
                        "filename": pdf_path.name,
                        "handin_month": handin_month,
                        "corrupt_cid": corrupt_cid,
                    }
                )

                if corrupt_cid:
                    logger.info(f"  Marked CID-corrupt (skipped): {pdf_path.name}")
                elif handin_month:
                    logger.info(f"  Found date in: {pdf_path.name} -> {handin_month}")
                else:
                    logger.info(f"  No date found in: {pdf_path.name}")

            except Exception as e:
                logger.error(f"  Error processing {pdf_path.name}: {e}")
                results.append(
                    {
                        "filename": pdf_path.name,
                        "handin_month": None,
                        "corrupt_cid": False,
                    }
                )

        return results

    def cleanup_batch(self, pdf_paths: list[Path]) -> None:
        for pdf_path in pdf_paths:
            try:
                pdf_path.unlink()
            except Exception as e:
                logger.warning(f"Failed to delete {pdf_path.name}: {e}")

    def process_all_batches(
        self, keep_pdfs: bool = False, max_batches: Optional[int] = None
    ) -> list[dict[str, str | bool | None]]:
        all_pdf_paths = self.list_thesis_pdfs()
        if not all_pdf_paths:
            logger.warning("No PDFs found in bucket!")
            return []

        all_results: list[dict[str, str | bool | None]] = []

        num_batches = (len(all_pdf_paths) + self.batch_size - 1) // self.batch_size
        if max_batches:
            num_batches = min(num_batches, max_batches)

        for batch_num in range(num_batches):
            start_idx = batch_num * self.batch_size
            logger.info(f"{'=' * 60}")
            logger.info(f"BATCH {batch_num + 1}/{num_batches}")
            logger.info(f"{'=' * 60}")

            local_paths = self.download_batch(all_pdf_paths, start_idx)
            batch_results = self.extract_from_batch(local_paths)
            all_results.extend(batch_results)

            if not keep_pdfs:
                self.cleanup_batch(local_paths)

        return all_results


def save_outputs(results: list[dict[str, str | bool | None]]) -> None:
    # Save CSV manually to avoid extra dependency; JSON for easy downstream processing.
    header = "filename;handin_month;corrupt_cid\n"
    rows = [
        (
            f"{row['filename']};"
            f"{'' if row['handin_month'] is None else row['handin_month']};"
            f"{row['corrupt_cid']}"
        )
        for row in results
    ]
    OUTPUT_CSV.write_text(header + "\n".join(rows) + "\n", encoding="utf-8")
    # OUTPUT_JSON.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")

In [85]:
# ==== SELECT MODE ====
print("Choose extraction mode: 'local' or 'gcp'")
while True:
    run_mode = input("Mode: 'local' or 'gcp'").strip().lower()
    export_mode = input("Export or not? [y/N]").strip().lower()
    if run_mode in {"local", "gcp"}:
        break
    print("Invalid mode. Please enter 'local' or 'gcp'.")

# ==== LOCAL MODE ====
if run_mode == "local":
    folder_path = Path("../Data/RAW_test/handin_test/")  # TESTING FOLDER
    pdf_files = sorted(folder_path.glob("*.pdf"))

    if not pdf_files:
        raise FileNotFoundError(f"No PDF files found in: {folder_path.resolve()}")

    total_files = len(pdf_files)
    available_filenames = {p.name: p for p in pdf_files}
    print(f"Found {total_files} PDF file(s) in {folder_path.resolve()}")

    while True:
        user_input = input(
            f"How many files do you want to process? (1-{total_files}, 'all', or a filename): "
        ).strip()

        user_input_lower = user_input.lower()

        if user_input_lower == "all":
            selected_pdf_files = pdf_files
            break

        if user_input_lower in {"q", "quit", "exit"}:
            raise SystemExit("Execution terminated by user. No files were processed.")

        if user_input in available_filenames:
            selected_pdf_files = [available_filenames[user_input]]
            break

        try:
            num_to_process = int(user_input)
        except ValueError:
            print(
                "Invalid input. Enter a positive integer, 'all', a valid filename, or 'q' to quit."
            )
            continue

        if 1 <= num_to_process <= total_files:
            selected_pdf_files = pdf_files[:num_to_process]
            break

        print(f"Please enter a value between 1 and {total_files}, 'all', or a valid filename.")

    print(f"Selected {len(selected_pdf_files)} file(s) for processing.")

    dates_exstracted = run_extraction(selected_pdf_files)
    print(f"Processed {len(dates_exstracted)} PDF file(s) from: {folder_path.resolve()}")

# ==== GCP MODE ====
else:
    bucket_name = GCP_DEFAULT_BUCKET
    gcs_prefix = GCP_DEFAULT_PREFIX
    print(f"Using default GCP source from gcp_metrics_from_pdfs.py: gs://{bucket_name}/{gcs_prefix}")

    batch_size_raw = input("Batch size [10]: ").strip()
    batch_size = int(batch_size_raw) if batch_size_raw else 10

    max_batches_raw = input("Max batches (blank = all): ").strip()
    max_batches = int(max_batches_raw) if max_batches_raw else None

    keep_pdfs_raw = input("Keep downloaded PDFs locally? [y/N]: ").strip().lower()
    keep_pdfs = keep_pdfs_raw in {"y", "yes"}

    processor = GCPBatchProcessor(
        bucket_name=bucket_name,
        gcs_prefix=gcs_prefix,
        local_pdf_folder="thesis_pdfs",
        batch_size=batch_size,
    )

    dates_exstracted = processor.process_all_batches(
        keep_pdfs=keep_pdfs,
        max_batches=max_batches,
    )

    print(f"Processed {len(dates_exstracted)} PDF file(s) from gs://{bucket_name}/{gcs_prefix}")

# ==== SHOW RESULTS ====
num_found = sum(1 for entry in dates_exstracted if entry["handin_month"])
num_total = len(dates_exstracted)
num_corrupt = sum(1 for entry in dates_exstracted if entry["corrupt_cid"])

print(
    f"\nExtracted hand-in month from {num_found}/{num_total} PDFs "
    f"({(num_found / num_total * 100) if num_total > 0 else 0:.1f}%)"
)
print(
    f"Marked CID-corrupt: {num_corrupt}/{num_total} PDFs "
    f"({(num_corrupt / num_total * 100) if num_total > 0 else 0:.1f}%)\n"
)

# ==== EXPORT RESULTS ====
if export_mode in {"y", "yes"}:
    save_outputs(dates_exstracted)
    print(f"CSV written to: {OUTPUT_CSV}")

Choose extraction mode: 'local' or 'gcp'
Using default GCP source from gcp_metrics_from_pdfs.py: gs://thesis_archive_bucket/dtu_findit/master_thesis/


2026-03-22 15:14:49,209 - seasonality_extractor - INFO - Initialising GCP Storage client...
/Users/oliver/Desktop/MSc_Speciale/Thesis_OCR/.venv/lib/python3.13/site-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/Users/oliver/Desktop/MSc_Speciale/Thesis_OCR/.venv/lib/python3.13/site-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD

KeyboardInterrupt: 

In [ ]:
display(dates_exstracted)

## JOIN SEASONALITY WITH METRICS

In [ ]:
# ==== MERGE WITH METRICS ANALYSIS ====
# Load the metrics df
candidate_paths = [
    Path("../Data/gcp_order/dtu_findit/extraction_and_processing/master_thesis_metrics_analysis.csv"),
    Path("Data/gcp_order/dtu_findit/extraction_and_processing/master_thesis_metrics_analysis.csv"),
]

csv_path = next((p for p in candidate_paths if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError("Could not find master_thesis_metrics_analysis.csv in expected locations.")

# File is semicolon-separated and contains some malformed rows.
metrics_df = pd.read_csv(csv_path, sep=";", engine="python")
print(f"Loaded: {csv_path}")
print(f"Shape: {metrics_df.shape}")

# Load the seasonality df
candidate_paths = [
    Path("../Data/gcp_order/dtu_findit/extraction_and_processing/handin_month_summary.csv"),
    Path("Data/gcp_order/dtu_findit/extraction_and_processing/handin_month_summary.csv"),
]

csv_path = next((p for p in candidate_paths if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError("Could not find handin_month_summary.csv in expected locations.")

# File is semicolon-separated and contains some malformed rows.
seasonality_df = pd.read_csv(csv_path, sep=";", engine="python", on_bad_lines="skip")
print(f"Loaded: {csv_path}")
print(f"Shape: {seasonality_df.shape}")

In [ ]:
# ==== MERGE DATAFRAMES ====
merged_dfs = pd.merge(
    metrics_df,
    seasonality_df,
    left_on="pdf_file",
    right_on="filename",
    how="inner",
)

# drop the redundant 'filename' column after the merge
merged_dfs.drop(columns=["filename"], inplace=True)

# rename column "handin_month" to "handin_month"
merged_dfs.rename(columns={"handin_month": "handin_month"}, inplace=True)

master_thesis_metrics_analysis_v2 = merged_dfs

print(type(master_thesis_metrics_analysis_v2))
display(len(master_thesis_metrics_analysis_v2))

# ==== EXPORT MERGED DATAFRAME ====
if export_mode in {"y", "yes"}:
    # export master_thesis_metrics_analysis_v2 to a new CSV file
    output_path = Path("Data/gcp_order/dtu_findit/extraction_and_processing/master_thesis_metrics_analysis_v2.csv")
    master_thesis_metrics_analysis_v2.to_csv(output_path, sep=";", index=False)
    print(f"Exported merged DataFrame to: {output_path}")
    

# ARCHIVES

In [56]:
# Open a PDF and print text from the first x pages

pdf_file_name = "5f46438bd9001d01721d7e29_Effect of probiotic TDA-producing bacteria on the microbiome of the aquaculture algae Isochrysis galbana (translated Eff.pdf"

pdf_path = "../Data/RAW_test/handin_test/"
pdf_input = pdf_path + pdf_file_name
x_pages = int(input("How many pages to print? ").strip())



if pdf_input:
    pdf_path = Path(pdf_input)
else:
    if "selected_pdf_files" in globals() and selected_pdf_files:
        pdf_path = selected_pdf_files[0]
    else:
        raise ValueError("No default PDF available. Provide a PDF path.")

if not pdf_path.exists():
    raise FileNotFoundError(f"PDF not found: {pdf_path}")

with pdfplumber.open(pdf_path) as pdf:
    total_pages = len(pdf.pages)
    n = min(x_pages, total_pages)
    print(f"Printing {n}/{total_pages} page(s) from: {pdf_path.name}\n")

    for i in range(n):
        text = pdf.pages[i].extract_text() or "[No extractable text]"
        print(f"{'=' * 20} PAGE {i + 1} {'=' * 20}")
        print(text)
        print()

Printing 4/95 page(s) from: 5f46438bd9001d01721d7e29_Effect of probiotic TDA-producing bacteria on the microbiome of the aquaculture algae Isochrysis galbana (translated Eff.pdf

==================== PAGE 1 ====================
[No extractable text]

==================== PAGE 2 ====================
[No extractable text]

==================== PAGE 3 ====================
[No extractable text]

==================== PAGE 4 ====================
[No extractable text]



## PRINT ALL RESULTS

In [ ]:
# print all results in a readable format
for entry in dates_exstracted:
    print(f"{entry['filename']}:\n{entry['handin_month']}\n")

In [ ]:
candidate_paths = [
    Path("../Data/gcp_order/dtu_findit/extraction_and_processing/master_thesis_metrics_analysis_v2.csv"),
    Path("Data/gcp_order/dtu_findit/extraction_and_processing/master_thesis_metrics_analysis_v2.csv"),
]

csv_path = next((p for p in candidate_paths if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError("Could not find master_thesis_metrics_analysis_v2.csv in expected locations.")

# File is semicolon-separated and contains some malformed rows.
master_thesis_metrics_analysis_v2 = pd.read_csv(csv_path, sep=";", engine="python", on_bad_lines="skip")
print(f"Loaded: {csv_path}")
print(f"Shape: {master_thesis_metrics_analysis_v2.shape}")
print(master_thesis_metrics_analysis_v2.columns)

In [ ]:
# fill page print
pd.set_option("display.max_colwidth", None)

# display pdf_file and handin_month columns where handin_month is not NaN
nan_df = master_thesis_metrics_analysis_v2[master_thesis_metrics_analysis_v2["handin_month"].isna()]
print(f"Entries with null handin_month: {len(nan_df)}")
display(nan_df[["pdf_file","member_id_ss_metrics", "handin_month"]].iloc[29:50])

## CHECK RUN NAN's

In [86]:
# fill page print
pd.set_option("display.max_colwidth", None)

# display dates_exstracted where handin_month is nan
extracted_df = pd.DataFrame(dates_exstracted)
extracted_non_null = extracted_df[extracted_df["handin_month"].isna()]
print(f"Extracted entries with non-null handin_month: {len(extracted_non_null)}")
print(f"Corrupt CID entries among non-null handin_month: {extracted_non_null['corrupt_cid'].sum()}")
print(f"Entries with null handin_month and marked as none-corrupt: {len(extracted_non_null[~extracted_non_null['corrupt_cid']])}")
display(extracted_non_null[~extracted_non_null["corrupt_cid"]])

Extracted entries with non-null handin_month: 0
Corrupt CID entries among non-null handin_month: 0
Entries with null handin_month and marked as none-corrupt: 0


,filename,handin_month,corrupt_cid


### FIXED FROM nan_df

- [x] 5ce67fe9d9001d016b4fccc7 - OCR with no space: **case 1**
- [x] 5d1c8d65d9001d146569a498 - OCR with DD-Month-YYYY: **case 2**
- [] 5d21d367d9001d2d454f054d - No year: "Projectperiod: 7. January-7. June" *risky to just exstract month and match with year from df*
- [x] 5d2b0defd9001d018a2333fc - case 1
- [x] 5d3d82f5d9001d32f558c096 - case 1
- [] 5d3d8373d9001d32f558c1a2 - Edge case, date given in raw text under acknowledgement. *difficult to exstract with precission*
- [] 5d3d838ad9001d32f558c1d6 - "June 28th, 2019": **case 3**
- [] 5d3d82e6d9001d32f558c081 - "ThethesiswasconductedfromJanuary2019toJune2019." *difficult to exstract with precission*

- [] ***BAD OCR***: "(cid:75)(cid:89)(cid:98)" - 5d1c8d79d9001d146569a4dc, 5d3d8395d9001d32f558c1ef, 
- [] ***NO DATE!*** - 5d3d836cd9001d32f558c193, 5d3d838ed9001d32f558c1e3, 5d3d83e5d9001d32f558c2a3, 5d4ff7e2d9001d1f8a6c41df

In [81]:

# ==== TEST: Date Format Coverage ====
# Verify patterns match and parsing works

test_cases = [
    ("2019-06-28", "June 2019"),
    ("2019/06/28", "June 2019"),
    ("2019.06.28", "June 2019"),
    ("20190628", "June 2019"),
    ("28062019", "June 2019"),
    ("Lyngby, 10-June-2019", "June 2019"),
    ("10-June-2019", "June 2019"),
    ("10/June/2019", "June 2019"),
    ("June-10-2019", "June 2019"),
    ("June/10/2019", "June 2019"),
    ("Oct. 12, 2019", "October 2019"),
    ("SEP. 2019", "September 2019"),
    ("okt. 2020", "October 2020"),
    ("jun. 2019", "June 2019"),
    ("June 10, 2019", "June 2019"),
    ("Lund, 10-July-2019", "July 2019"),
    ("10-July-2019, Lund", "July 2019"),
    ("Lund,10-July-2019", "July 2019"),
    ("Kongens Lyngby Campus 324, 10-July-2019", "July 2019"),
    ("10-July-2019, Kongens Lyngby Campus 324", "July 2019"),
    ("Period 27January2025to27July2025", "July 2025"),
    ("27January2025to27July2025", "July 2025"),
    ("27thofJune,2020", "June 2020"),
    ("Period27thofJune,2020to27thofJuly,2020", "July 2020"),
    ("Lund,10July2020", "July 2020"),
    ("Lund10July2020", "July 2020"),
    ("June 28th, 2019", "June 2019"),
    ("June 1st, 2019", "June 2019"),
    ("June 22nd, 2019", "June 2019"),
    ("June 23rd, 2019", "June 2019"),
    ("28th of June, 19", "June 2019"),
    ("June '19", "June 2019"),
    ("10 June 2019", "June 2019"),
    ("April23,2019", "April 2019"),
    ("29/10/2018-29/04/2019", "April 2019"),
    ("June,2020", "June 2020"),
    ("June-2020", "June 2020"),
    ("June/2020", "June 2020"),
    ("June.2020", "June 2020"),
    ("2020 June", "June 2020"),
    ("2020-June", "June 2020"),
    ("05 2020", "May 2020"),
    ("05-2020", "May 2020"),
    ("2020-05", "May 2020"),
    ("2020/05", "May 2020"),
    ("2020.05", "May 2020"),
    ("2020 05", "May 2020"),
    ("28th ofJune,2019", "June 2019"),
    ("28th of June 2019", "June 2019"),
    ("28th-of-June-2019", "June 2019"),
    ("28th/of/June/2019", "June 2019"),
    ("28thofJune2019", "June 2019"),
    ("Period 28th ofJune,2019 to 29th ofJuly,2019", "July 2019"),
    ("January 2019 to June 2019", "June 2019"),
    ("Jan. 2019 - Jun. 2019", "June 2019"),
    ("from March 2020 until July 2020", "July 2020"),
    ("Marts 2019 til juni 2019", "June 2019"),
    ("June of 2019", "June 2019"),
    ("in June of 2019", "June 2019"),
]

print("Testing date extraction on various formats:\n")
failures = []
for test_text, expected in test_cases:
    result = extract_handin_month(test_text)
    passed = result == expected
    status = "✓" if passed else "✗"
    pretty_result = result if result is not None else "None"
    print(f"{status} '{test_text}' -> {pretty_result:15} (expected: {expected})")
    if not passed:
        failures.append((test_text, result, expected))

if failures:
    print(f"\n{len(failures)} FAILURES:")
    for test_text, result, expected in failures:
        print(f"  '{test_text}': got '{result}', expected '{expected}'")
else:
    print(f"\n✓ ALL {len(test_cases)} TESTS PASSED!")


Testing date extraction on various formats:

✓ '2019-06-28' -> June 2019       (expected: June 2019)
✓ '2019/06/28' -> June 2019       (expected: June 2019)
✓ '2019.06.28' -> June 2019       (expected: June 2019)
✓ '20190628' -> June 2019       (expected: June 2019)
✓ '28062019' -> June 2019       (expected: June 2019)
✓ 'Lyngby, 10-June-2019' -> June 2019       (expected: June 2019)
✓ '10-June-2019' -> June 2019       (expected: June 2019)
✓ '10/June/2019' -> June 2019       (expected: June 2019)
✓ 'June-10-2019' -> June 2019       (expected: June 2019)
✓ 'June/10/2019' -> June 2019       (expected: June 2019)
✓ 'Oct. 12, 2019' -> October 2019    (expected: October 2019)
✓ 'SEP. 2019' -> September 2019  (expected: September 2019)
✓ 'okt. 2020' -> October 2020    (expected: October 2020)
✓ 'jun. 2019' -> June 2019       (expected: June 2019)
✓ 'June 10, 2019' -> June 2019       (expected: June 2019)
✓ 'Lund, 10-July-2019' -> July 2019       (expected: July 2019)
✓ '10-July-2019, Lund' 

# CURL TO DOWNLOAD files in nan_df

In [76]:
# Compact verification for all current test cases
failures = []
for test_text, expected in test_cases:
    got = extract_handin_month(test_text)
    if got != expected:
        failures.append((test_text, got, expected))

print("Total tests:", len(test_cases))
print("Failures:", len(failures))
if failures:
    for item in failures[:10]:
        print("FAIL:", item)

print("\nFocused new-format checks:")
for s in ["June,2020", "June-2020", "June/2020", "2020 June", "05-2020", "2020 05"]:
    print(s, "->", extract_handin_month(s))

Total tests: 59
Failures: 0

Focused new-format checks:
June,2020 -> June 2020
June-2020 -> June 2020
June/2020 -> June 2020
2020 June -> June 2020
05-2020 -> May 2020
2020 05 -> May 2020
